# gRNA negative control selection  
Use bowtie to map negative controls to the genome(GRCh38), tolerating mismatches

In [ ]:
import pandas as pd
from Bio import SeqIO

sam_columns = [
    "QNAME",  # Query name (read ID)
    "FLAG",   # Bitwise flag
    "RNAME",  # Reference sequence name
    "POS",    # Position
    "MAPQ",   # Mapping quality
    "CIGAR",  # CIGAR string
    "RNEXT",  # Mate reference sequence name
    "PNEXT",  # Mate position
    "TLEN",   # Template length
    "SEQ",    # Query sequence
    "QUAL",    # Base quality
	"XA",
	"MD",
	"NM",
	"XM"
]

def read_sam(sam_path):
	df = pd.read_csv(sam_path, sep="\t", comment="@", names=sam_columns, header=None)
	df['QNAME'] = df['QNAME'].astype(str)
	return df

In [ ]:
def negative_control_selection(perfect_match_sam_path, imperfect_match_sam_path, imperfect_match_unmapped_fa_path, output_path, mismatch_tolerance):
	perfect_match_sam = read_sam(perfect_match_sam_path)
	imperfect_match_sam = read_sam(imperfect_match_sam_path)

	perfect_match_sam_AAVS1 = perfect_match_sam[
		(perfect_match_sam['QNAME'].str.startswith("AAVS1")) &  
		(perfect_match_sam['RNAME'] == "chr19")                 
	]
	# there are two AAVS1 gRNAs that have multiple mappings on chromosome 19

	keys = ['QNAME', 'RNAME', 'POS', 'CIGAR']  # Adjust columns to ensure uniqueness
	merged = pd.merge(
		imperfect_match_sam, 
		perfect_match_sam_AAVS1[keys], 
		on=keys, 
		how='left', 
		indicator=True
	)
	print(len(merged))
	imperfect_match_sam_nonAAVS = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
	print(len(imperfect_match_sam_nonAAVS))

	original_qnames = set(imperfect_match_sam['QNAME'])
	filtered_qnames = set(imperfect_match_sam_nonAAVS['QNAME'])
	imperfect_match_unmapped_AAVS = list(original_qnames - filtered_qnames)

	imperfect_match_unmapped = [
		record.id for record in SeqIO.parse(imperfect_match_unmapped_fa_path, "fasta")
		]
	
	print(len(imperfect_match_unmapped_AAVS))
	print(len(imperfect_match_unmapped))
	
	# selected negative controls:
	# imperfect_match_unmapped + imperfect_match_mapped (AAVS1, only 1 mapping to the original region)
	# save as txt
	final_list = imperfect_match_unmapped + imperfect_match_unmapped_AAVS
	with open(output_path, "w") as output_file:
		for qname in final_list:
			output_file.write(qname + "\n")

In [ ]:
mismatch_tolerance = 2
mismatch_tolerance_str = str(mismatch_tolerance)

negative_control_selection(
	perfect_match_sam_path = '/media/scratch/fy2306/projects/base_editing/data/bowtie/negative_controls.bowtie_hg38.mismatch0.sam',
	imperfect_match_sam_path = f'/media/scratch/fy2306/projects/base_editing/data/bowtie/negative_controls.bowtie_hg38.mismatch{mismatch_tolerance_str}.sam',
	imperfect_match_unmapped_fa_path = f'/media/scratch/fy2306/projects/base_editing/data/bowtie/negative_controls.bowtie_hg38.mismatch{mismatch_tolerance_str}.unmapped.fa',
	output_path = f'/media/scratch/fy2306/projects/base_editing/data/bowtie/negative_controls.clean.mismatch{mismatch_tolerance_str}.txt',
	mismatch_tolerance = mismatch_tolerance
)